# Urdu Aegis voice — LoRA fine-tune for English loanwords

**One self-contained notebook.** Put the `aegis-urdu-loanword` folder in your
Google Drive (dataset/, ur-aegis-female/, aegis-female.ckpt), open this in
Colab, set **Runtime → Change runtime type → GPU (T4)**, then **Run all**.

Teaches `ur_PK-aegis_female-medium` to say *keypad / card / transaction / OTP*
inside Urdu by training tiny low-rank adapters on the **frozen** model
(~1% of weights), then folding them into the weights → a normal Piper ONNX.
Every base weight stays byte-identical, so plain Urdu can't regress.

## You need the base checkpoint

Piper trains from a `.ckpt`. Put the **Aegis student checkpoint** at
`/content/aegis-female.ckpt` before running (Files pane, or `hf_hub_download`).
The notebook stops if it is absent — there is no fallback.
**Cell 5b plays the base voice before training — confirm it is clean there.**

**`dataset.zip`** must contain `metadata.csv` (lines `wav/0001.wav|<urdu text>`,
`|`-delimited) and a `wav/` folder of 22 050 Hz mono WAVs. Cell 3 either
unzips an uploaded one, or (fallback) rebuilds it from Uplift AI if you
paste a key.

The bundled set is **185 loanword-dense clips (~9.5 min)** — the original 123 plus 62 that widen coverage of dental ت/د vs retroflex ٹ/ڈ, long vowels (iː aː eː oː), and bank-IVR vocabulary (verification, installment, refund, beneficiary, …). `SENTENCES` in cell 3 mirrors `metadata.csv` 1:1 for the Uplift rebuild path.

Runtime ~30-45 min on a T4. Outputs: `ur_PK-aegis_female_loan-medium.onnx`
+ `.json` (cell 9 also copies them to `Drive/loan-out/`; rename to
`ur_PK-aegis_female-medium` for the final drop-in ship).

## 1 · GPU

In [5]:
import torch
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> GPU'
print(torch.cuda.get_device_name(0), '| torch', torch.__version__)

Tesla T4 | torch 2.11.0+cu128


## 1b · Google Drive

Everything the run needs — `dataset/`, `ur-aegis-female/` (the ONNX), and
`aegis-female.ckpt` — lives in one Drive folder. Mount it once; nothing to upload.

In [6]:
from google.colab import drive
import pathlib
drive.mount("/content/drive")

# shared folder: https://drive.google.com/drive/folders/1lxKm-MJOjDl6PT0K2GgVAeAL3YDN8Eyk
# needs: dataset/ (metadata.csv + wav/) and ur-aegis-female/ (the .onnx + .json).
# The trainable .ckpt is rebuilt from the ONNX in cell 5 -- nothing else to upload.
DRIVE = pathlib.Path("/content/drive/MyDrive/aegis-urdu-loanword")
assert DRIVE.is_dir(), (
    f"{DRIVE} not found. Put the aegis-urdu-loanword folder at the top of My "
    "Drive (or a shortcut to it there), or edit this path.")
print("Drive folder:", ", ".join(sorted(p.name for p in DRIVE.iterdir())))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive folder: ab, aegis-female.ckpt, dataset, fasih.ckpt, loan-out, tools, ur-aegis-female


## 2 · Install piper1-gpl (training) — ~6-8 min

`scikit-build` etc. must be present **before** the build (it compiles `piper.espeakbridge`, which the training phonemiser needs).

In [7]:
%%bash
set -e
apt-get -qq update >/dev/null 2>&1 || true
apt-get -qq install -y build-essential cmake ninja-build >/dev/null 2>&1 || true
pip -q install scikit-build cmake ninja 'cython>=3,<4' espeakng-loader

[ -d /content/piper1-gpl ] || git clone -q https://github.com/OHF-voice/piper1-gpl.git /content/piper1-gpl
pip -q install -e '/content/piper1-gpl[train]'
cd /content/piper1-gpl
bash ./build_monotonic_align.sh
python setup.py -q build_ext --inplace

# new torch ONNX exporter (dynamo) can't trace piper's stochastic duration
# predictor -> force the legacy exporter (idempotent, self-repairing)
python3 - <<'PY'
import re, pathlib
_p = pathlib.Path("/content/piper1-gpl/src/piper/train/export_onnx.py")
_s = _p.read_text()
_s = re.sub(r"(dynamo=False,\s*)+", "", _s)          # strip any prior insert(s)
_s = _s.replace("torch.onnx.export(", "torch.onnx.export(dynamo=False, ", 1)
_p.write_text(_s)
print("export_onnx.py: dynamo=False set (once)")
PY

pip -q install onnx onnxscript huggingface_hub soundfile

# --- GUARANTEE a complete espeak-ng-data --------------------------------------
# piper1-gpl builds espeak-ng from source (scikit-build) and copies
# espeak-ng-data into the package. On a fresh Colab that copy is unreliable; a
# PARTIAL espeak-ng-data silently truncates phonemisation -> every sentence
# becomes a ~10-phoneme stub -> 0.6 s renders and a model trained on fragments.
# Overlay a known-good data dir from the `espeakng-loader` wheel when needed.
python3 - <<'PY'
import shutil, pathlib, piper
pkg = pathlib.Path(piper.__file__).parent
dst = pkg / "espeak-ng-data"
if not ((dst / "ur_dict").is_file() and (dst / "phontab").is_file()):
    import espeakng_loader
    src = pathlib.Path(espeakng_loader.get_data_path())
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"espeak-ng-data: overlaid from {src}")
else:
    print(f"espeak-ng-data: already complete at {dst}")
assert (dst / "ur_dict").is_file() and (dst / "phontab").is_file(), \
    "espeak-ng-data has no Urdu -- cannot continue"
PY

echo "--- imports + Urdu phonemiser hard check ---"
python3 - <<'PY'
import piper.espeakbridge
from piper.train.vits.monotonic_align.core import maximum_path_c
from piper.train.vits.lightning import VitsModel
from piper.train.vits.dataset import VitsDataModule
from piper.phonemize_espeak import EspeakPhonemizer
import piper, pathlib
d = str(pathlib.Path(piper.__file__).parent / "espeak-ng-data")
out = EspeakPhonemizer(d).phonemize("ur", "آپ کے کارڈ پر ایک فراڈ ٹرانزیکشن ہے۔")
n = sum(len(x) for x in out)
print(f"  ur -> {n} phonemes  {out}")
assert n >= 30, (
    f"espeak-ng Urdu is BROKEN on this runtime ({n} phonemes, expect ~50). "
    "Do not train. Runtime > Disconnect and delete runtime, then Run all.")
print("training imports OK + Urdu phonemiser OK")
PY


Compiling /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx because it changed.
[1/1] Cythonizing /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx
[1/2] Copying espeak-ng-data after espeak-ng external project builds
[1/2] Install the project...
-- Install configuration: "Release"
-- Up-to-date: /content/piper1-gpl/_skbuild/linux-x86_64-3.13/cmake-install/src/piper/./espeakbridge.so
copying _skbuild/linux-x86_64-3.13/cmake-install/src/piper/espeakbridge.so -> src/piper/espeakbridge.so

export_onnx.py: dynamo=False set (once)
espeak-ng-data: already complete at /content/piper1-gpl/src/piper/espeak-ng-data
--- imports + Urdu phonemiser hard check ---
  ur -> 51 phonemes  [['ˌ', 'a', 'ː', 'p', ' ', 'k', 'e', 'ː', ' ', 'k', 'ˈ', 'a', 'ː', 'r', 'ɖ', ' ', 'p', 'ˈ', 'ʌ', 'r', ' ', 'ˈ', 'e', 'ː', 'k', ' ', 'f', 'r', 'ˈ', 'a', 'ː', 'ɖ', ' ', 'ʈ', 'r', 'ˌ', 'a', 'ː', 'n', 'z', 'ˈ', 'e', 'ː', 'k', 'ʃ', 'a', 'n', ' ', 'h', 'ɛ', '.']]
training imports OK + Urdu phone

/usr/local/lib/python3.13/dist-packages/Cython/Compiler/Main.py:381: FutureWarning: Cython directive 'language_level' not set, using '3str' for now (Py3). This has changed from earlier releases! File: /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx
  tree = Parsing.p_module(s, pxd, full_module_name)
performance hint: core.pyx:7:5: Exception check on 'maximum_path_each' will always require the GIL to be acquired.
Possible solutions:
	1. Declare 'maximum_path_each' as 'noexcept' if you control the definition and you're sure you don't want the function to raise exceptions.
	2. Use an 'int' return type on 'maximum_path_each' to allow an error code to be returned.
performance hint: core.pyx:38:6: Exception check on 'maximum_path_c' will always require the GIL to be acquired.
Possible solutions:
	1. Declare 'maximum_path_c' as 'noexcept' if you control the definition and you're sure you don't want the function to raise exceptions.
	2. Use an 'int' return type on 'maximum_pa

In [8]:
pip -q install -e '/content/piper1-gpl[train]'


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for piper-tts (pyproject.toml) ... done


## 3 · Dataset

Copied from `DRIVE/dataset/` (metadata.csv + wav/). Falls back to an
uploaded `dataset.zip`, or a rebuild from Uplift AI if you paste a key.


In [9]:
# leave blank to use an uploaded dataset.zip; paste a key to (re)build it
UPLIFT_API_KEY = ""          # sk_api_...  (optional fallback)
UPLIFT_VOICE   = "helpdesk-agent"

In [10]:
SENTENCES = [
"حفاظت کے لیے، براہِ کرم اپنے شناختی کارڈ کے آخری چھ ہندسے اب اپنے کی پیڈ پر درج کریں۔",
"براہِ کرم اپنے کارڈ کے آخری چار ہندسے اب اپنے کی پیڈ پر درج کریں، انہیں بول کر نہ بتائیں۔",
"براہِ کرم اپنی تاریخِ پیدائش سال، مہینہ، دن کی ترتیب میں اپنے کی پیڈ پر درج کریں۔",
"میرے کارڈ پر ایک فراڈ ٹرانزیکشن ہے، براہِ کرم اسے بلاک کر دیں۔",
"میرے کریڈٹ کارڈ پر کتنی رقم واجب الادا ہے؟",
"مجھے نئی چیک بک چاہیے، براہِ کرم میرے رجسٹرڈ پتے پر بھیج دیں۔",
"میرا کارڈ گم ہو گیا ہے، اسے فوراً بلاک کر دیں۔",
"میں آپ سے آپ کا پن، سی وی وی، او ٹی پی یا پاس ورڈ کبھی نہیں پوچھوں گی۔",
"آپ کا کارڈ بلاک کر دیا گیا ہے اور آپ کے موبائل نمبر پر ایس ایم ایس بھیج دیا گیا ہے۔",
"آپ کے اکاؤنٹ کا بیلنس پچاس ہزار روپے ہے۔",
"آپ کے کریڈٹ کارڈ کی لمٹ پانچ لاکھ روپے ہے اور دستیاب رقم چودہ ہزار روپے ہے۔",
"آپ کی کمپلینٹ رجسٹرڈ ہو گئی ہے، ریفرنس نمبر نوٹ کر لیں۔",
"آپ کی ٹرانسفر کی ریکوئسٹ پراسیس ہو رہی ہے۔",
"براہِ کرم ایچ بی ایل موبائل ایپ کھولیں اور آن لائن اسٹیٹمنٹ دیکھیں۔",
"آپ کا اے ٹی ایم کارڈ ایکسپائر ہو چکا ہے، نیا کارڈ برانچ سے وصول کریں۔",
"آپ کے کارڈ کی تفصیل اسکرین پر موجود ہے۔",
"آپ کے اکاؤنٹ کی تفصیل اسکرین پر موجود ہے۔",
"آپ کے بیلنس کی تفصیل اسکرین پر موجود ہے۔",
"آپ کے اسٹیٹمنٹ کی تفصیل اسکرین پر موجود ہے۔",
"آپ کے ٹرانزیکشن کی تفصیل اسکرین پر موجود ہے۔",
"آپ کے لمٹ کی تفصیل اسکرین پر موجود ہے۔",
"براہِ کرم اپنا پن کسی کے ساتھ شیئر نہ کریں۔",
"براہِ کرم اپنا او ٹی پی کسی کے ساتھ شیئر نہ کریں۔",
"براہِ کرم اپنا سی وی وی کسی کے ساتھ شیئر نہ کریں۔",
"براہِ کرم اپنا پاس ورڈ کسی کے ساتھ شیئر نہ کریں۔",
"براہِ کرم اپنا کوڈ کسی کے ساتھ شیئر نہ کریں۔",
"میں آپ کے اکاؤنٹ کی جانچ کر رہی ہوں، براہِ کرم انتظار کریں۔",
"میں آپ کے کارڈ کی جانچ کر رہی ہوں، براہِ کرم انتظار کریں۔",
"میں آپ کے ٹرانزیکشن کی جانچ کر رہی ہوں، براہِ کرم انتظار کریں۔",
"میں آپ کے بیلنس کی جانچ کر رہی ہوں، براہِ کرم انتظار کریں۔",
"میں آپ کے سٹیٹس کی جانچ کر رہی ہوں، براہِ کرم انتظار کریں۔",
"میں آپ کے اسٹیٹمنٹ کی جانچ کر رہی ہوں، براہِ کرم انتظار کریں۔",
"آپ کا موبائل کامیابی سے اپڈیٹ ہو گیا ہے۔",
"آپ کا ای میل کامیابی سے اپڈیٹ ہو گیا ہے۔",
"آپ کا پن کامیابی سے اپڈیٹ ہو گیا ہے۔",
"آپ کا پاس ورڈ کامیابی سے اپڈیٹ ہو گیا ہے۔",
"آپ کے کارڈ پر کوئی مشکوک سرگرمی نظر نہیں آئی۔",
"آپ کے اکاؤنٹ پر کوئی مشکوک سرگرمی نظر نہیں آئی۔",
"آپ کے والیٹ پر کوئی مشکوک سرگرمی نظر نہیں آئی۔",
"آپ کا کارڈ بلاک کر دیا گیا ہے اور نیا جاری کر دیا گیا ہے۔",
"آپ کا اے ٹی ایم بلاک کر دیا گیا ہے اور نیا جاری کر دیا گیا ہے۔",
"آپ ایپ کے ذریعے بھی یہ کام کر سکتے ہیں۔",
"آپ آن لائن کے ذریعے بھی یہ کام کر سکتے ہیں۔",
"آپ انٹرنیٹ کے ذریعے بھی یہ کام کر سکتے ہیں۔",
"آپ ایس ایم ایس کے ذریعے بھی یہ کام کر سکتے ہیں۔",
"آپ اے ٹی ایم کے ذریعے بھی یہ کام کر سکتے ہیں۔",
"آپ کال کے ذریعے بھی یہ کام کر سکتے ہیں۔",
"آپ کی کمپلینٹ موصول ہو گئی ہے اور اس پر کارروائی جاری ہے۔",
"آپ کا ٹرانزیکشن چارج پانچ سو روپے ہے۔",
"آپ کا سروس چارج پانچ سو روپے ہے۔",
"آپ کا کارڈ چارج پانچ سو روپے ہے۔",
"براہِ کرم اپنا کی پیڈ دوبارہ چیک کریں۔",
"آپ کا کی پیڈ تیار ہے۔",
"یہ کی پیڈ محفوظ رکھیں۔",
"میں آپ کو کی پیڈ کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا کارڈ دوبارہ چیک کریں۔",
"آپ کا کارڈ تیار ہے۔",
"یہ کارڈ محفوظ رکھیں۔",
"میں آپ کو کارڈ کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا او ٹی پی دوبارہ چیک کریں۔",
"آپ کا او ٹی پی تیار ہے۔",
"یہ او ٹی پی محفوظ رکھیں۔",
"میں آپ کو او ٹی پی کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا سی این آئی سی دوبارہ چیک کریں۔",
"آپ کا سی این آئی سی تیار ہے۔",
"یہ سی این آئی سی محفوظ رکھیں۔",
"میں آپ کو سی این آئی سی کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا ایچ بی ایل دوبارہ چیک کریں۔",
"آپ کا ایچ بی ایل تیار ہے۔",
"یہ ایچ بی ایل محفوظ رکھیں۔",
"میں آپ کو ایچ بی ایل کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا اے ٹی ایم دوبارہ چیک کریں۔",
"آپ کا اے ٹی ایم تیار ہے۔",
"یہ اے ٹی ایم محفوظ رکھیں۔",
"میں آپ کو اے ٹی ایم کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا ایس ایم ایس دوبارہ چیک کریں۔",
"آپ کا ایس ایم ایس تیار ہے۔",
"یہ ایس ایم ایس محفوظ رکھیں۔",
"میں آپ کو ایس ایم ایس کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا پن دوبارہ چیک کریں۔",
"آپ کا پن تیار ہے۔",
"یہ پن محفوظ رکھیں۔",
"میں آپ کو پن کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا سی وی وی دوبارہ چیک کریں۔",
"آپ کا سی وی وی تیار ہے۔",
"یہ سی وی وی محفوظ رکھیں۔",
"میں آپ کو سی وی وی کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا ٹرانزیکشن دوبارہ چیک کریں۔",
"آپ کا ٹرانزیکشن تیار ہے۔",
"یہ ٹرانزیکشن محفوظ رکھیں۔",
"میں آپ کو ٹرانزیکشن کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا بیلنس دوبارہ چیک کریں۔",
"آپ کا بیلنس تیار ہے۔",
"یہ بیلنس محفوظ رکھیں۔",
"میں آپ کو بیلنس کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا اسٹیٹمنٹ دوبارہ چیک کریں۔",
"آپ کا اسٹیٹمنٹ تیار ہے۔",
"یہ اسٹیٹمنٹ محفوظ رکھیں۔",
"میں آپ کو اسٹیٹمنٹ کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا پاس ورڈ دوبارہ چیک کریں۔",
"آپ کا پاس ورڈ تیار ہے۔",
"یہ پاس ورڈ محفوظ رکھیں۔",
"میں آپ کو پاس ورڈ کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا ایپ دوبارہ چیک کریں۔",
"آپ کا ایپ تیار ہے۔",
"یہ ایپ محفوظ رکھیں۔",
"میں آپ کو ایپ کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا کریڈٹ کارڈ دوبارہ چیک کریں۔",
"آپ کا کریڈٹ کارڈ تیار ہے۔",
"یہ کریڈٹ کارڈ محفوظ رکھیں۔",
"میں آپ کو کریڈٹ کارڈ کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا ڈیبٹ کارڈ دوبارہ چیک کریں۔",
"آپ کا ڈیبٹ کارڈ تیار ہے۔",
"یہ ڈیبٹ کارڈ محفوظ رکھیں۔",
"میں آپ کو ڈیبٹ کارڈ کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا برانچ دوبارہ چیک کریں۔",
"آپ کا برانچ تیار ہے۔",
"یہ برانچ محفوظ رکھیں۔",
"میں آپ کو برانچ کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا ٹرانسفر دوبارہ چیک کریں۔",
"آپ کا ٹرانسفر تیار ہے۔",
"یہ ٹرانسفر محفوظ رکھیں۔",
"میں آپ کو ٹرانسفر کے بارے میں بتاتی ہوں۔",
"آپ کی درخواست کی تاریخ اور ٹرانزیکشن کی تفصیل ریکارڈ کر لی گئی ہے۔",
"براہِ کرم درست تاریخِ پیدائش درج کریں تاکہ ویریفیکیشن مکمل ہو سکے۔",
"آپ کے کریڈٹ کارڈ کی ادائیگی کی آخری تاریخ اکتیس مارچ ہے۔",
"دستیاب رقم اور کریڈٹ لمٹ کی تفصیل اسکرین پر موجود ہے۔",
"تصدیق کے بعد آپ کا ڈیبٹ کارڈ ایکٹیویٹ کر دیا جائے گا۔",
"توثیق مکمل ہونے پر آپ کی ٹرانسفر کی درخواست پراسیس ہو جائے گی۔",
"آپ کے اکاؤنٹ کی تفصیل میں پتہ اور تاریخِ پیدائش دونوں اپڈیٹ ہو گئے ہیں۔",
"درج کردہ ٹرانزیکشن آئی ڈی کی مدت ختم ہو چکی ہے، نئی درخواست دیں۔",
"آپ کی اسٹیٹمنٹ کی مدت یکم تا تیس دن مقرر کی گئی ہے۔",
"دفتری اوقات میں آپ کی کمپلینٹ کا تدارک کر دیا جائے گا۔",
"براہِ کرم تاخیر سے بچنے کے لیے ڈیو ڈیٹ سے پہلے ادائیگی کریں۔",
"آپ کے ڈیبٹ کارڈ اور کریڈٹ کارڈ کی تاریخِ اجرا ایک ہی دن ہے۔",
"تناسب کے مطابق مارک اپ آپ کی اگلی اسٹیٹمنٹ میں درج ہو گا۔",
"توثیقی کوڈ درج کریں، یہ کوڈ صرف دو منٹ کے لیے درست رہے گا۔",
"آپ کے اعتماد کا شکریہ، آپ کا ٹرانزیکشن محفوظ طریقے سے مکمل ہوا۔",
"درخواست کی تصدیق کے بعد نیا اے ٹی ایم کارڈ متعلقہ برانچ بھیج دیا جائے گا۔",
"آپ کی ادائیگی کی درست تاریخ اور ٹرانزیکشن کی رقم دونوں ریکارڈ پر ہیں۔",
"تفصیلی اسٹیٹمنٹ کے لیے تاریخوں کی حد درج کریں۔",
"براہِ کرم اپنی درج کردہ ای میل کی توثیق کریں تاکہ نوٹیفیکیشن موصول ہوں۔",
"کارڈ کی تبدیلی کے بعد اسٹینڈنگ انسٹرکشن خودبخود منتقل ہو جائے گی۔",
"براہِ کرم اپنے کی پیڈ پر چار ہندسوں کا پاس کوڈ درج کریں۔",
"آپ کی درج کردہ ای میل پر ایک تصدیقی لنک بھیج دیا گیا ہے۔",
"آپ کے اسکرین پر دی گئی ریفرنس نمبر نوٹ کر لیں۔",
"آج مارکیٹ ریٹ کے مطابق آپ کے اکاؤنٹ میں رقم منتقل ہو گئی ہے۔",
"آپ کے کارڈ پر لاگو سالانہ چارج پانچ ہزار روپے ہے۔",
"پیمنٹ گیٹ وے سے جواب موصول ہونے تک براہِ کرم انتظار کریں۔",
"آپ کا کریڈٹ کارڈ بیلنس اور کم از کم ادائیگی اسکرین پر موجود ہے۔",
"نیا کوڈ حاصل کرنے کے لیے دوبارہ ری سینڈ کا بٹن دبائیں۔",
"سیکیورٹی کی وجہ سے آپ کا سیشن ختم کر دیا گیا ہے، دوبارہ لاگ اِن کریں۔",
"آپ کی درخواست پر کارڈ کی لمٹ میں اضافہ منظور کر لیا گیا ہے۔",
"روٹ تبدیل ہونے کی صورت میں نیا ریفرنس نمبر جاری کیا جائے گا۔",
"مینو میں سے بیلنس انکوائری کا آپشن منتخب کریں۔",
"آپ کے پوائنٹس کی ویلیو اگلی اسٹیٹمنٹ میں کیش بیک کے طور پر ظاہر ہو گی۔",
"اپنے موبائل ایپ کی سیٹنگز میں جا کر نوٹیفیکیشن آن کر لیں۔",
"آپ کا اے ٹی ایم پن ری سیٹ کرنے کے لیے پرانا پن درج کریں۔",
"براہِ کرم اسکرین پر دیا گیا کیو آر کوڈ اسکین کریں۔",
"آپ کی درخواست موڈ تبدیل ہونے کے بعد ری اپلائی کریں۔",
"آن لائن پورٹل پر لاگ اِن کر کے ای اسٹیٹمنٹ ڈاؤن لوڈ کریں۔",
"آپ کی ویریفیکیشن کامیابی سے مکمل ہو گئی ہے۔",
"بایومیٹرک تصدیق کے لیے براہِ کرم اپنا انگوٹھا اسکینر پر رکھیں۔",
"آپ کے فنگر پرنٹ کا ریکارڈ سسٹم میں موجود نہیں، برانچ سے رجوع کریں۔",
"نیا بینیفیشری شامل کر دیا گیا ہے اور چوبیس گھنٹے بعد فعال ہو گا۔",
"آپ کا ٹرانزیکشن مرچنٹ کی جانب سے ڈیکلائن کر دیا گیا ہے۔",
"پی او ایس مشین پر ادائیگی منظور ہو گئی ہے۔",
"آپ کی قسط یعنی انسٹالمنٹ اکتیس تاریخ کو واجب الادا ہے۔",
"اس ٹرانزیکشن پر مارک اپ کی شرح دو فیصد ماہانہ ہے۔",
"آپ کا ری فنڈ سات کاروباری دنوں میں اکاؤنٹ میں واپس آ جائے گا۔",
"غلط ٹرانزیکشن کی ریورسل کی درخواست موصول ہو گئی ہے۔",
"آپ کا ڈسپیوٹ کیس کھول دیا گیا ہے، ریفرنس نمبر محفوظ رکھیں۔",
"آپ کو ہر ٹرانزیکشن پر ایس ایم ایس الرٹ اور ایپ نوٹیفیکیشن موصول ہوں گے۔",
"منی اسٹیٹمنٹ میں صرف آخری دس ٹرانزیکشنز دکھائی جاتی ہیں۔",
"آپ کا کارڈ عارضی طور پر ہولڈ پر رکھ دیا گیا ہے۔",
"کارڈ دوبارہ چالو یعنی ری ایکٹیویٹ کرنے کے لیے تصدیقی کوڈ درج کریں۔",
"آٹو ڈیبٹ کی سہولت آپ کے اکاؤنٹ پر فعال کر دی گئی ہے۔",
"آپ کی اسٹینڈنگ انسٹرکشن ہر مہینے کی پہلی تاریخ کو عمل میں آئے گی۔",
"ادائیگی کی تصدیق یعنی کنفرمیشن آپ کے موبائل پر بھیج دی گئی ہے۔",
"درخواست منسوخ کرنے کے لیے کینسل کا بٹن دبائیں۔",
"فارم مکمل کرنے کے بعد سبمٹ کا بٹن دبائیں۔",
"آپ کی لاگ اِن کوشش ناکام رہی، براہِ کرم ری ٹرائی کریں۔",
"سیشن ٹائم آؤٹ ہونے پر آپ کو دوبارہ لاگ اِن کرنا ہو گا۔",
"آپ کا ورچوئل کارڈ فوری طور پر جاری کر دیا گیا ہے۔",
"فزیکل کارڈ آپ کے رجسٹرڈ پتے پر دس دن میں پہنچ جائے گا۔"
]


In [11]:
import pathlib, zipfile, shutil, wave

CANON = pathlib.Path("/content/dataset")          # canonical layout: metadata.csv + wav/
(CANON / "wav").mkdir(parents=True, exist_ok=True)

def _have_dataset():
    m = CANON / "metadata.csv"
    if not m.is_file():
        return False
    n = len([x for x in m.read_text(encoding="utf-8").splitlines() if x.strip()])
    return n > 0 and len(list((CANON / "wav").glob("*.wav"))) >= n

if not _have_dataset() and "DRIVE" in globals() and (DRIVE / "dataset" / "metadata.csv").is_file():
    _src = DRIVE / "dataset"
    (CANON / "wav").mkdir(parents=True, exist_ok=True)
    shutil.copy(_src / "metadata.csv", CANON / "metadata.csv")
    for _w in (_src / "wav").glob("*.wav"):
        shutil.copy(_w, CANON / "wav" / _w.name)
    print("dataset from Drive:", len(list((CANON / "wav").glob("*.wav"))), "wavs")

if not _have_dataset():
    # 1. get a zip
    z = pathlib.Path("/content/dataset.zip")
    if not z.is_file():
        try:
            from google.colab import files
            print("Upload dataset.zip (metadata.csv + wav/ inside)…")
            z = pathlib.Path("/content") / list(files.upload())[0]
        except Exception:
            z = None
    if z and z.is_file():
        tmp = pathlib.Path("/content/_dsraw"); shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir()
        with zipfile.ZipFile(z) as zf:
            zf.extractall(tmp)
        meta = next(iter(tmp.rglob("metadata.csv")), None)
        assert meta is not None, "no metadata.csv in the zip"
        src_root = meta.parent
        rows = []
        for line in meta.read_text(encoding="utf-8").splitlines():
            if "|" not in line:
                continue
            rel, text = line.split("|", 1)
            cand = (src_root / rel)
            if not cand.is_file():
                cand = next(iter(src_root.rglob(pathlib.Path(rel).name)), None)
            assert cand and cand.is_file(), f"missing wav for {rel}"
            name = pathlib.Path(rel).name
            shutil.copy(cand, CANON / "wav" / name)
            rows.append(f"wav/{name}|{text.strip()}")
        (CANON / "metadata.csv").write_text("\n".join(rows) + "\n", encoding="utf-8")
        shutil.rmtree(tmp, ignore_errors=True)

# 2. Uplift fallback
if not _have_dataset() and UPLIFT_API_KEY.startswith("sk_"):
    import requests
    rows = []
    for i, t in enumerate(SENTENCES, 1):
        r = requests.post("https://api.upliftai.org/v1/synthesis/text-to-speech",
            headers={"Authorization": f"Bearer {UPLIFT_API_KEY}"},
            json={"voiceId": UPLIFT_VOICE, "text": t, "outputFormat": "WAV_22050_16"}, timeout=90)
        r.raise_for_status()
        (CANON / "wav" / f"{i:04d}.wav").write_bytes(r.content)
        rows.append(f"wav/{i:04d}.wav|{t}")
        if i % 20 == 0: print(f"  {i}/{len(SENTENCES)}")
    (CANON / "metadata.csv").write_text("\n".join(rows) + "\n", encoding="utf-8")

assert _have_dataset(), "no dataset — put dataset/ on Drive, upload dataset.zip, or paste an Uplift key"
DATASET_DIR = str(CANON)
_rows = [l for l in (CANON / "metadata.csv").read_text(encoding="utf-8").splitlines() if l.strip()]
_wr = {w.name: wave.open(str(w)).getframerate() for w in (CANON / "wav").glob("*.wav")}
_wrong = {n: r for n, r in _wr.items() if r != 22050}
assert not _wrong, f"WAVs must be 22050 Hz -- offenders: {dict(list(_wrong.items())[:5])}"
_d = sum(wave.open(str(w)).getnframes() / 22050 for w in (CANON / "wav").glob("*.wav"))
print(f"dataset OK: {len(_rows)} clips at {DATASET_DIR}  ({_d/60:.1f} min)")
print("  first line:", _rows[0][:70])


# --- sanity: what does the training corpus actually sound like? ------------
# The adapter cannot beat its teacher. If these clips are muffled / robotic,
# that timbre is the ceiling and no training hyperparameter fixes it.
import random, wave as _wave, numpy as _np
from IPython.display import Audio as _Audio, display as _display, Markdown as _Md
_mrows = [l for l in (CANON / "metadata.csv").read_text(encoding="utf-8").splitlines() if l.strip()]
for _l in random.Random(0).sample(_mrows, 3):
    _rel, _txt = _l.split("|", 1)
    _wf = _wave.open(str(CANON / _rel.strip()))
    _pcm = _np.frombuffer(_wf.readframes(_wf.getnframes()), _np.int16)
    _display(_Md(f"`{_txt.strip()}`  ({_wf.getframerate()} Hz, {_wf.getnframes()/_wf.getframerate():.1f}s)"))
    _display(_Audio(_pcm, rate=_wf.getframerate()))


dataset OK: 185 clips at /content/dataset  (9.5 min)
  first line: wav/0001.wav|حفاظت کے لیے، براہِ کرم اپنے شناختی کارڈ کے آخری چھ ہندسے


`میں آپ کو اسٹیٹمنٹ کے بارے میں بتاتی ہوں۔`  (22050 Hz, 2.3s)

`براہِ کرم اپنا کریڈٹ کارڈ دوبارہ چیک کریں۔`  (22050 Hz, 2.8s)

`آپ کے کریڈٹ کارڈ کی لمٹ پانچ لاکھ روپے ہے اور دستیاب رقم چودہ ہزار روپے ہے۔`  (22050 Hz, 4.7s)

## 4 · Base model (Aegis ONNX — phonemisation + config)

In [12]:
import shutil, pathlib
_src = (DRIVE / "ur-aegis-female") if "DRIVE" in globals() else pathlib.Path("/nonexistent")
if (_src / "ur_PK-aegis_female-medium.onnx").is_file():
    shutil.copy(_src / "ur_PK-aegis_female-medium.onnx", "/content/aegis.onnx")
    shutil.copy(_src / "ur_PK-aegis_female-medium.onnx.json", "/content/aegis.onnx.json")
    print("Aegis ONNX + config from Drive")
else:
    from huggingface_hub import hf_hub_download
    shutil.copy(hf_hub_download("mahwizzzz/piper-voice-ur-aegis-female",
        "ur_PK-aegis_female-medium.onnx"), "/content/aegis.onnx")
    shutil.copy(hf_hub_download("mahwizzzz/piper-voice-ur-aegis-female",
        "ur_PK-aegis_female-medium.onnx.json"), "/content/aegis.onnx.json")
    print("Aegis ONNX + config from Hugging Face")


Aegis ONNX + config from Drive


## 4b · Phoneme sanity — the corpus must phonemise to real Urdu

The #1 silent failure: an incomplete espeak-ng-data makes every sentence collapse to a ~10-phoneme stub, so the model trains on fragments and every render is ~0.6 s. This gate catches it **before** 20 minutes of training.

In [13]:
# phonemise real corpus lines; a stub result == espeak-ng Urdu is broken.
import json as _json
from piper import PiperVoice as _PV

_pidS = _json.load(open("/content/aegis.onnx.json"))["phoneme_id_map"]
_pvS  = _PV.load("/content/aegis.onnx")
_rowsS = [l for l in open(f"{DATASET_DIR}/metadata.csv", encoding="utf-8") if l.strip()]
_badS = []
for _l in _rowsS[:8]:
    _txt = _l.split("|", 1)[1].strip()
    _flat = [p for s in _pvS.phonemize(_txt) for p in s]
    _drop = sorted(set(p for p in _flat if p not in _pidS))
    _ok = len(_flat) >= 0.4 * len(_txt) and not _drop
    print(f"{'ok ' if _ok else 'BAD'} {len(_flat):3d} phonemes / {len(_txt):3d} chars  drop={_drop}  {_txt[:40]}")
    if not _ok: _badS.append(_txt)
del _pvS
assert not _badS, (
    f"{len(_badS)} line(s) phonemise to a stub / drop phonemes -- espeak-ng Urdu is "
    "broken on this runtime. Re-run section 2 (install); if it still fails, "
    "Runtime > Disconnect and delete runtime, then Run all. DO NOT train on this.")
print("\nphoneme sanity OK -- full Urdu phonemisation")


ok  139 phonemes /  85 chars  drop=[]  حفاظت کے لیے، براہِ کرم اپنے شناختی کارڈ
ok  137 phonemes /  89 chars  drop=[]  براہِ کرم اپنے کارڈ کے آخری چار ہندسے اب
ok  123 phonemes /  81 chars  drop=[]  براہِ کرم اپنی تاریخِ پیدائش سال، مہینہ،
ok   92 phonemes /  62 chars  drop=[]  میرے کارڈ پر ایک فراڈ ٹرانزیکشن ہے، براہ
ok   62 phonemes /  42 chars  drop=[]  میرے کریڈٹ کارڈ پر کتنی رقم واجب الادا ہ
ok   96 phonemes /  61 chars  drop=[]  مجھے نئی چیک بک چاہیے، براہِ کرم میرے رج
ok   68 phonemes /  46 chars  drop=[]  میرا کارڈ گم ہو گیا ہے، اسے فوراً بلاک ک
ok  104 phonemes /  70 chars  drop=[]  میں آپ سے آپ کا پن، سی وی وی، او ٹی پی ی

phoneme sanity OK -- full Urdu phonemisation


## 5 · Base checkpoint — built from the Aegis ONNX

Piper trains from a `.ckpt`; Aegis ships only the inference ONNX. This cell
rebuilds a trainable checkpoint by grafting **every** ONNX weight onto a fresh
piper1-gpl generator — the whole inference path: text encoder, stochastic
duration predictor, residual-coupling flow, HiFi-GAN vocoder. It is verified
**bit-exact** against the ONNX (max sample diff ~1e-4 at zero noise).

`enc_q`, the discriminator and the duration predictor's training-only sublayers
are not in the ONNX and start from fresh init — they don't touch inference, and
the LoRA run freezes the base anyway.

If you already have a real Aegis student `.ckpt`, drop it at
`/content/aegis-female.ckpt` and this cell keeps it as-is.

**Cell 5b then plays it — it must be clean, natural, female before you train.**

In [14]:
# Rebuild a PURE Female-Aegis trainable .ckpt from /content/aegis.onnx.
import os, sys, json, warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, "/content/piper1-gpl/src")
import torch

ONNX = "/content/aegis.onnx"
CKPT = "/content/aegis-female.ckpt"
CFG  = json.load(open(ONNX + ".json"))            # num_symbols / phoneme_id_map, used later

# piper "medium" == ModelAudioConfig.low_quality(); the rest are piper defaults.
ARCH = dict(spec_channels=513, segment_size=8192, inter_channels=192,
            hidden_channels=192, filter_channels=768, n_heads=2, n_layers=6,
            kernel_size=3, p_dropout=0.1, resblock="2",
            resblock_kernel_sizes=(3, 5, 7),
            resblock_dilation_sizes=((1, 2), (2, 6), (3, 12)),
            upsample_rates=(8, 8, 4), upsample_initial_channel=256,
            upsample_kernel_sizes=(16, 16, 8),
            n_speakers=1, gin_channels=0, use_sdp=True)


def build_ckpt_from_onnx(onnx_path, out_path):
    import onnx
    from onnx import numpy_helper
    from piper.train.vits.models import SynthesizerTrn

    g = onnx.load(onnx_path)
    inits = {t.name: torch.from_numpy(numpy_helper.to_array(t).copy())
             for t in g.graph.initializer}
    named = {n: w for n, w in inits.items() if not n.startswith("onnx::")}

    # weight-norm-folded convs: destination param comes from the ONNX NODE NAME,
    # never from graph order (the exporter traces the flow in reverse ->
    # flows.6 before flows.0; ordering by position swaps the flow blocks and
    # the voice becomes pure noise).
    opaque = []
    for node in g.graph.node:
        if node.op_type not in ("Conv", "ConvTranspose"):
            continue
        oin = [i for i in node.input if i.startswith("onnx::") and i in inits]
        if oin:
            key = node.name.strip("/").rsplit("/", 1)[0].replace("/", ".") + ".weight_v"
            opaque.append((key, oin[0]))

    net = SynthesizerTrn(n_vocab=CFG["num_symbols"], **ARCH)
    sd = net.state_dict()
    new = {}

    for n, w in named.items():
        if n in sd and tuple(sd[n].shape) == tuple(w.shape):
            new[n] = w.to(sd[n].dtype)
    assert len(new) == len(named), f"named graft {len(new)}/{len(named)}"

    _exps = [en for en in inits if en.startswith("onnx::Exp_")]   # SDP logs
    assert len(_exps) == 1, f"expected 1 onnx::Exp_ (dp.flows.0.logs), found {_exps}"
    new["dp.flows.0.logs"] = (-inits[_exps[0]]).reshape(
        sd["dp.flows.0.logs"].shape).to(sd["dp.flows.0.logs"].dtype)

    for tgt, oname in opaque:
        w = inits[oname]
        assert tuple(sd[tgt].shape) == tuple(w.shape), f"{tgt}: {tuple(sd[tgt].shape)} vs {tuple(w.shape)}"
        new[tgt] = w.to(sd[tgt].dtype)
        gk = tgt[:-1] + "g"                           # weight_v -> weight_g
        gv = torch.linalg.vector_norm(w.reshape(w.shape[0], -1), dim=1)
        new[gk] = gv.reshape(sd[gk].shape).to(sd[gk].dtype)
    assert len(opaque) >= 53, f"only {len(opaque)} opaque convs"

    full = dict(sd); full.update(new)                 # enc_q/disc/post_* stay fresh-init
    torch.save({"state_dict": {f"model_g.{k}": v for k, v in full.items()},
                "global_step": 0, "epoch": 0,
                "pytorch-lightning_version": "2.0.0", "hyper_parameters": {}},
               out_path)
    print(f"grafted {len(named)} named + {len(opaque)} weight-norm convs "
          f"-> {out_path}  ({os.path.getsize(out_path)/1e6:.0f} MB)")


import shutil
# 100% pure Female Aegis: always rebuild from the ONNX (deterministic, ~30 s).
# A stale /content/*.ckpt from an earlier run would silently poison training.
for _stale in (CKPT, "/content/merged.ckpt", "/content/fasih.ckpt"):
    if os.path.exists(_stale):
        os.remove(_stale)
build_ckpt_from_onnx(ONNX, CKPT)

_st = torch.load(CKPT, map_location="cpu")["state_dict"]
assert any(k.startswith("model_g.enc_p.") for k in _st), "bad checkpoint"
print(f"pure Aegis-female checkpoint: {len(_st)} tensors, "
      f"{os.path.getsize(CKPT)/1e6:.0f} MB")
print("enc_q + discriminator are training-only and start from init; "
      "enc_q is trained (cell 7), both are discarded at export.")
print("-> run cell 5b: base voice must be clean female speech.")
del _st


grafted 296 named + 53 weight-norm convs -> /content/aegis-female.ckpt  (95 MB)
pure Aegis-female checkpoint: 673 tensors, 95 MB
enc_q + discriminator are training-only and start from init; enc_q is trained (cell 7), both are discarded at export.
-> run cell 5b: base voice must be clean female speech.


## 5b · Test the base checkpoint *before* training

Synthesise a few lines straight from `/content/aegis-female.ckpt` with no
adapter. This must be **clean, natural female speech**. If it is noise, the
checkpoint is wrong: stop here, don't train on it.

In [15]:
import numpy as np, gc, torch
from piper import PiperVoice
from piper.train.vits.lightning import VitsModel
from IPython.display import Audio, display, Markdown

_pid = CFG["phoneme_id_map"]
_pv  = PiperVoice.load("/content/aegis.onnx")            # phonemisation only

_probe = VitsModel(num_symbols=CFG["num_symbols"], num_speakers=1,
                   sample_rate=22050, batch_size=8, mos_metric="none")
_m, _u = _probe.load_state_dict(
    torch.load("/content/aegis-female.ckpt", map_location="cpu")["state_dict"],
    strict=False)
print(f"loaded base: {len(_m)} missing, {len(_u)} unexpected")
_probe.model_g.eval()

for _t in ["آج میں آپ کی کیا مدد کر سکتی ہوں؟",
           "اپنے فون کے کی پیڈ پر چار ہندسے دبائیں۔",
           "آپ کے کارڈ پر ایک فراڈ ٹرانزیکشن ہے۔"]:
    _flat = [x for s in _pv.phonemize(_t) for x in s]
    _ids = [1]
    for _p in _flat:
        if _p in _pid: _ids += _pid[_p] + [0]
    _ids += [2]
    assert len(_flat) >= 0.4 * len(_t), (
        f"{_t!r} phonemised to {len(_flat)} phonemes -- espeak-ng Urdu is broken, "
        "STOP (see section 2 / the phoneme-sanity cell).")
    with torch.no_grad():
        _a = _probe.model_g.infer(torch.LongTensor([_ids]), torch.LongTensor([len(_ids)]),
                noise_scale=0.667, length_scale=1.0, noise_scale_w=0.8)[0][0, 0].numpy()
    display(Markdown(f"`{_t}`  ({len(_flat)} phonemes, {len(_ids)} ids, {len(_a)/22050:.1f}s)"))
    display(Audio(_a, rate=22050))

del _probe, _pv; gc.collect()
try: torch.cuda.empty_cache()
except Exception: pass
print("clean speech -> continue.  noise -> the ckpt is bad, do not train.")

loaded base: 111 missing, 0 unexpected


`آج میں آپ کی کیا مدد کر سکتی ہوں؟`  (49 phonemes, 100 ids, 2.6s)

`اپنے فون کے کی پیڈ پر چار ہندسے دبائیں۔`  (59 phonemes, 120 ids, 2.8s)

`آپ کے کارڈ پر ایک فراڈ ٹرانزیکشن ہے۔`  (51 phonemes, 104 ids, 2.6s)

clean speech -> continue.  noise -> the ckpt is bad, do not train.


## 6 · The low-rank adapter

In [16]:
# Low-rank adapters for a piper1-gpl VITS model.
# Train ONLY these; every base weight stays frozen and byte-identical, so
# plain-Urdu output cannot regress. At export they fold into the weights ->
# a normal Piper ONNX. set_lora_scale(0) == the exact original voice.
import torch
import torch.nn as nn

_LORA_CLSNAME = "LoRAConv1d"


def _is_lora(m):
    # NOT isinstance(): re-running this cell in a notebook rebinds the class, so
    # adapters injected by an earlier run would fail an isinstance() check and
    # set_lora_scale / merge_lora would silently become no-ops -- exactly the
    # "|dm_p|/|m_p| = 0.000, the scale sweep does nothing" symptom.
    return getattr(m, "_lora_marker", False) is True or type(m).__name__ == _LORA_CLSNAME


class LoRAConv1d(nn.Module):
    """Frozen base Conv1d + trainable low-rank branch (up-proj zero-init)."""

    _lora_marker = True

    def __init__(self, base: nn.Conv1d, rank: int = 8, alpha: float = 8.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.rank = rank
        self.base_scaling = alpha / rank
        self.scaling = self.base_scaling
        self.enabled = True
        k = base.kernel_size[0]
        pad = base.padding[0] if isinstance(base.padding, tuple) else base.padding
        self.lora_down = nn.Conv1d(base.in_channels, rank, k, stride=base.stride,
                                   padding=pad, dilation=base.dilation, bias=False)
        self.lora_up = nn.Conv1d(rank, base.out_channels, 1, bias=False)
        nn.init.kaiming_uniform_(self.lora_down.weight, a=5 ** 0.5)
        nn.init.zeros_(self.lora_up.weight)

    def forward(self, x):
        out = self.base(x)
        if self.enabled:
            out = out + self.lora_up(self.lora_down(x)) * self.scaling
        return out

    @torch.no_grad()
    def merged_weight(self):
        up = self.lora_up.weight.squeeze(-1)
        delta = torch.einsum("or,rik->oik", up, self.lora_down.weight)
        return self.base.weight + delta * self.scaling


def _iter_conv1d(model):
    for name, mod in model.named_modules():
        if _is_lora(mod):
            continue                              # don't descend into an adapter
        for cn, ch in list(mod.named_children()):
            if cn in ("base", "lora_down", "lora_up"):
                continue                          # never re-wrap adapter internals
            if isinstance(ch, nn.Conv1d) and not _is_lora(ch):
                yield mod, cn, f"{name}.{cn}".lstrip(".")


def _iter_lora(model):
    for name, mod in model.named_modules():
        for cn, ch in list(mod.named_children()):
            if _is_lora(ch):
                yield mod, cn


def inject_lora(model, targets, rank=8, alpha=8.0):
    n = 0
    for parent, cn, dotted in list(_iter_conv1d(model)):
        if any(t in dotted for t in targets):
            setattr(parent, cn, LoRAConv1d(getattr(parent, cn), rank, alpha))
            n += 1
    return n


def freeze_base(model):
    for name, p in model.named_parameters():
        p.requires_grad_(".lora_down." in name or ".lora_up." in name)


def lora_parameters(model):
    return [p for n, p in model.named_parameters()
            if ".lora_down." in n or ".lora_up." in n]


def set_lora_scale(model, scale):
    n = 0
    for m in model.modules():
        if _is_lora(m):
            m.enabled = scale != 0.0
            m.scaling = scale * m.base_scaling
            n += 1
    assert n > 0, "set_lora_scale touched 0 adapters -- did cell 7 run?"
    return n


@torch.no_grad()
def merge_lora(model):
    for parent, cn in list(_iter_lora(model)):
        lora = getattr(parent, cn)
        conv = lora.base
        conv.weight.data.copy_(lora.merged_weight())
        setattr(parent, cn, conv)
    return model


## 7 · Train (base frozen, only the adapter learns)

In [17]:
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, TQDMProgressBar, Callback
from piper.train.vits.lightning import VitsModel
from piper.train.vits.dataset import VitsDataModule

# ===== config -- sweep these ============================================
RANK      = 8          # adapter rank (capacity)
ALPHA     = 8          # adapter strength; effective gain = ALPHA / RANK
LR        = 1e-4       # adapter learning rate
ENCQ_LR   = 1e-4       # posterior-encoder LR (trained from scratch, discarded at export)
MAX_STEPS = 1000
BATCH     = 8
TARGETS   = ["enc_p.encoder.attn_layers", "enc_p.encoder.ffn_layers", "enc_p.proj"]
#   drop "enc_p.proj" to leave the latent prior (mean/variance) exactly Aegis
#   -- that is the layer whose drift shows up as a muffled / robotic voice.
# =======================================================================


class LossLog(Callback):
    """Print the raw loss components every N steps -- the telemetry the last
    run was missing. Watch: kl should fall then flatten; mel should keep
    drifting down; if kl is still diving at MAX_STEPS the adapter is chasing a
    moving target (enc_q not converged)."""

    def __init__(self, every=25):
        self.every = every

    def on_train_batch_end(self, trainer, *_):
        s = trainer.global_step
        if s == 0 or s % self.every:
            return
        m = trainer.callback_metrics
        g = lambda k: (float(m[k]) if k in m else float("nan"))
        print(f"step {s:4d} | mel {g('train_mel'):6.3f} | kl {g('train_kl'):8.3f} "
              f"| dur {g('train_dur'):8.2f} | fm {g('train_fm'):6.2f} "
              f"| gen {g('train_gen'):5.2f} | disc {g('train_disc'):5.2f}", flush=True)


model = VitsModel(num_symbols=CFG["num_symbols"], num_speakers=1,
                  sample_rate=22050, batch_size=BATCH, learning_rate=LR,
                  mos_metric="none")   # UTMOS download loops + breaks rich on Colab
# load the base checkpoint into the PLAIN model, THEN wrap the convs
miss, unexp = model.load_state_dict(torch.load("/content/aegis-female.ckpt",
                                    map_location="cpu")["state_dict"], strict=False)
_bad = [k for k in miss if not k.startswith("model_d.")]
print(f"warmstart: {len(miss)} missing ({len(_bad)} outside model_d), {len(unexp)} unexpected")
# every missing key should be the discriminator (model_d): training-only, not in
# the Aegis ONNX. VitsModel inits it, opt_d trains it. That is expected.
assert not _bad and not unexp, f"unexpected checkpoint gaps: {_bad[:5]} / {unexp[:5]}"

n = inject_lora(model.model_g, TARGETS, RANK, ALPHA)
freeze_base(model.model_g)
# enc_q has no Aegis weights (not in the ONNX). Train it: the mel loss runs
# it -> frozen female dec, so it learns female latents, which become the KL
# target the LoRA adapters fit. enc_q is discarded at export.
for _p in model.model_g.enc_q.parameters():
    _p.requires_grad_(True)
trn = sum(p.numel() for p in lora_parameters(model.model_g))
encq = sum(p.numel() for p in model.model_g.enc_q.parameters())
print(f"LoRA: {n} convs, {trn} adapter params + {encq} enc_q params (trained, then discarded)")
print(f"config: RANK={RANK} ALPHA={ALPHA} gain={ALPHA/RANK:.2f} LR={LR} "
      f"ENCQ_LR={ENCQ_LR} MAX_STEPS={MAX_STEPS} TARGETS={TARGETS}")

_orig = model.configure_optimizers
def _co():
    opts, scheds = _orig(); opts, scheds = list(opts), list(scheds)
    g = torch.optim.AdamW(
        [{"params": lora_parameters(model.model_g), "lr": model.hparams.learning_rate},
         {"params": list(model.model_g.enc_q.parameters()), "lr": ENCQ_LR}],
        lr=model.hparams.learning_rate, betas=model.hparams.betas, eps=model.hparams.eps)
    opts[0] = g
    scheds[0] = torch.optim.lr_scheduler.ExponentialLR(g, gamma=model.hparams.lr_decay)
    return opts, scheds
model.configure_optimizers = _co

dm = VitsDataModule(csv_path=f"{DATASET_DIR}/metadata.csv",
    audio_dir=f"{DATASET_DIR}", cache_dir="/content/cache",
    espeak_voice="ur", config_path="/content/out.onnx.json",
    voice_name="ur_PK-aegis_female", sample_rate=22050,
    num_symbols=CFG["num_symbols"], batch_size=BATCH)

_mc = ModelCheckpoint(dirpath="/content/train/ckpts", save_last=True,
                      monitor="val_mel", mode="min", save_top_k=1)
trainer = L.Trainer(accelerator="gpu", devices=1, precision="16-mixed",
    max_steps=MAX_STEPS, default_root_dir="/content/train",
    log_every_n_steps=25, num_sanity_val_steps=0, enable_model_summary=False,
    callbacks=[TQDMProgressBar(refresh_rate=10),   # NOT rich (recursion-crashes on Colab)
               LossLog(25), _mc])
trainer.fit(model, dm)
print("training done")
try:
    print(f"best val_mel = {float(_mc.best_model_score):.4f}  @  {_mc.best_model_path}")
    print("(cell 9 merges the FINAL step, not this best checkpoint -- if best is "
          "much better, reload it there before merging.)")
except Exception:
    pass


INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


warmstart: 111 missing (0 outside model_d), 0 unexpected
LoRA: 37 convs, 262656 adapter params + 7238016 enc_q params (trained, then discarded)
config: RANK=8 ALPHA=8 gain=1.00 LR=0.0001 ENCQ_LR=0.0001 MAX_STEPS=1000 TARGETS=['enc_p.encoder.attn_layers', 'enc_p.encoder.ffn_layers', 'enc_p.proj']


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

step 1000 | mel  0.832 | kl    4.455 | dur     2.39 | fm   5.15 | gen  3.02 | disc  1.50


Validation: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_steps=1000` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=1000` reached.


training done
best val_mel = 0.6942  @  /content/train/ckpts/epoch=19-step=800.ckpt
(cell 9 merges the FINAL step, not this best checkpoint -- if best is much better, reload it there before merging.)


## 8 · A/B — plain Urdu must be unchanged, loanwords should improve

In [18]:
# --- free the training footprint before loading onnxruntime + more GPU work.
#     1000 steps of VITS + the discriminator + two AdamW optimiser states left
#     resident is what OOM-kills the kernel in this cell. ------------------
import gc, torch
for _n in ("trainer", "dm", "_co", "_orig"):
    globals().pop(_n, None)
try:
    model.zero_grad(set_to_none=True)
except Exception:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    _free, _tot = torch.cuda.mem_get_info()
    print(f"GPU free after cleanup: {_free/1e9:.1f} / {_tot/1e9:.1f} GB")

import numpy as np, pathlib, wave, shutil
from piper import PiperVoice
from IPython.display import Audio, display, Markdown

PLAIN = ["آج میں آپ کی کیا مدد کر سکتی ہوں؟", "آپ کا شکریہ، خدا حافظ۔"]
# held-out loanword lines (NOT verbatim in metadata.csv) exercising the v2
# enrichment axes: retroflex ʈ/ɖ, dental-vs-retroflex contrast, long vowels,
# and the wider bank-IVR vocabulary.
LOAN  = ["براہِ کرم اپنے موبائل کے کی پیڈ سے پرانا پن مٹا دیں۔",
         "اس ٹرانزیکشن کی تصدیق کے لیے ایک دبائیں۔",
         "آپ کے کریڈٹ کارڈ کا بیلنس صفر ہو چکا ہے۔",
         "نئی اسٹیٹمنٹ آپ کی ای میل پر بھیج دی گئی ہے۔",
         "آپ کا ڈیبٹ کارڈ آج اپڈیٹ کر دیا گیا ہے۔",
         "درج کردہ تاریخ اور کریڈٹ کارڈ نمبر دوبارہ چیک کریں۔",
         "آپ کی ویریفیکیشن اور اگلی انسٹالمنٹ دونوں مکمل ہیں۔",
         "اسکرین پر دیا گیا ریفرنس نمبر نوٹ کر لیں۔"]
pid = CFG["phoneme_id_map"]
pv = PiperVoice.load("/content/aegis.onnx")
phon = {t: pv.phonemize(t) for t in PLAIN + LOAN}
model.model_g.eval()

# scales to audition: 0.0 == frozen base, 0.33 == the ship candidate.
# 1.0 was consistently over-cooked, so it is not rendered.
SCALES = (0.0, 0.33)
PROBE  = SCALES[-1]

def _ids_of(t):
    ids = [1]
    for p in (x for s in phon[t] for x in s):
        if p in pid: ids += pid[p] + [0]
    return ids + [2]

def _infer(ids):
    with torch.no_grad():
        return model.model_g.infer(
            torch.LongTensor([ids]).to(model.device),
            torch.LongTensor([len(ids)]).to(model.device),
            noise_scale=0.667, length_scale=1.0, noise_scale_w=0.8)[0][0, 0].float().cpu().numpy()

def synth(scale):
    set_lora_scale(model.model_g, scale)
    return {t: _infer(_ids_of(t)) for t in PLAIN + LOAN}

# phonemiser sanity -- an empty id list -> a silent 0:00 clip in the player
for t in PLAIN[:1] + LOAN[:1]:
    assert len(_ids_of(t)) > 4, f"phonemiser made {len(_ids_of(t))} ids for {t!r} (id-map mismatch)"
print(f"phonemiser OK ({len(_ids_of(LOAN[0]))} ids for the first loanword line)")

# --- probe: does the adapter collapse the latent prior? -----------------
# A muffled / "dabi hui" voice almost always == the prior std exp(logs_p)
# pushed down at enc_p.proj. Compare base (0.0) vs the ship scale:
#   std ratio well below 1.0  ->  drop "enc_p.proj" from TARGETS in cell 7.
#   |dm_p|/|m_p| == 0.000      ->  the adapter never trained (see cell 6/7).
print(f"\nprior stats     base -> adapter@{PROBE}")
for t in PLAIN[:1] + LOAN[:2]:
    ids = torch.LongTensor([_ids_of(t)]).to(model.device)
    ln  = torch.LongTensor([ids.shape[1]]).to(model.device)
    with torch.no_grad():
        set_lora_scale(model.model_g, 0.0);   _, m0, ls0, _ = model.model_g.enc_p(ids, ln)
        set_lora_scale(model.model_g, PROBE); _, m1, ls1, _ = model.model_g.enc_p(ids, ln)
    print(f"  {t[:20]:22s} std {ls0.exp().mean():.3f} -> {ls1.exp().mean():.3f} "
          f"(x{ls1.exp().mean()/ls0.exp().mean():.2f})   "
          f"|dm_p|/|m_p| {(m1-m0).norm()/m0.norm():.3f}")

# --- scale sweep (free -- no retraining) --------------------------------
renders = {s: synth(s) for s in SCALES}

# scale-0 must equal the real Aegis voice (frozen base) or the A/B is meaningless
_ref = np.concatenate([np.frombuffer(x.audio_int16_bytes, np.int16).astype(np.float32) / 32768
                       for x in pv.synthesize(PLAIN[0])])
def _lm(x, n=1024, h=256):
    w = np.hanning(n)
    return np.log(np.abs(np.array([np.fft.rfft(w * x[j:j+n]) for j in range(0, len(x)-n, h)])) + 1e-6)
_b = renders[0.0][PLAIN[0]]; _k = min(len(_b), len(_ref))
_d = float(np.abs(_lm(_b[:_k]) - _lm(_ref[:_k])).mean()) if _k > 2048 else 9.9
print(f"\nbase-vs-Aegis logmel L1 = {_d:.2f}  (<~2 = frozen base intact; >3 = corrupted)")

# --- dump every render to disk (Colab audio widgets sometimes show 0:00) -
DUMP = pathlib.Path("/content/ab"); DUMP.mkdir(exist_ok=True)
def _wav(path, x):
    x = np.nan_to_num(np.asarray(x, np.float32))
    pk = float(np.abs(x).max()) or 1.0
    with wave.open(str(path), "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(22050)
        w.writeframes((x / pk * 0.95 * 32767).astype("<i2").tobytes())

for i, t in enumerate(PLAIN + LOAN):
    tag = "PLAIN" if t in PLAIN else "LOAN"
    r0 = renders[0.0][t]
    k = min(len(r0), len(renders[PROBE][t]))
    l1 = np.abs(r0[:k] - renders[PROBE][t][:k]).mean()
    display(Markdown(f"**{tag}**  L1({PROBE} vs base) = {l1:.3f} — `{t}`"))
    for s in SCALES:
        a = renders[s][t]
        _wav(DUMP / f"{i:02d}_{tag}_scale{s}.wav", a)
        bad = "  ⚠ silent/NaN" if (not np.isfinite(a).all() or np.abs(a).max() < 1e-4) else ""
        display(Markdown(f"&nbsp;&nbsp;scale {s}  ({len(a)/22050:.1f}s){bad}"))
        display(Audio(a, rate=22050))

if "DRIVE" in globals():
    (DRIVE / "ab").mkdir(exist_ok=True)
    for f in DUMP.glob("*.wav"):
        shutil.copy(f, DRIVE / "ab" / f.name)
    print(f"\nWAVs dumped to {DUMP} and {DRIVE / 'ab'}")
else:
    print(f"\nWAVs dumped to {DUMP}")

print("\nWhat to look for:")
print(" - PLAIN rows: L1 small AND still sounds like Aegis at scale 0.33.")
print(" - LOAN rows:  the loanword should be clearer at 0.33 than at 0.0.")
print(" - scale 0.33 muffled -> lower ALPHA / drop enc_p.proj / fewer steps (cell 7).")


GPU free after cleanup: 15.0 / 15.6 GB
phonemiser OK (160 ids for the first loanword line)

prior stats     base -> adapter@0.33
  آج میں آپ کی کیا مدد   std 1.026 -> 1.038 (x1.01)   |dm_p|/|m_p| 0.125
  براہِ کرم اپنے موبائ   std 1.027 -> 1.042 (x1.01)   |dm_p|/|m_p| 0.137
  اس ٹرانزیکشن کی تصدی   std 1.026 -> 1.039 (x1.01)   |dm_p|/|m_p| 0.128

base-vs-Aegis logmel L1 = 1.29  (<~2 = frozen base intact; >3 = corrupted)


**PLAIN**  L1(0.33 vs base) = 0.128 — `آج میں آپ کی کیا مدد کر سکتی ہوں؟`

&nbsp;&nbsp;scale 0.0  (2.7s)

&nbsp;&nbsp;scale 0.33  (2.3s)

**PLAIN**  L1(0.33 vs base) = 0.088 — `آپ کا شکریہ، خدا حافظ۔`

...

&nbsp;&nbsp;scale 0.33  (4.1s)

**LOAN**  L1(0.33 vs base) = 0.152 — `اسکرین پر دیا گیا ریفرنس نمبر نوٹ کر لیں۔`

&nbsp;&nbsp;scale 0.0  (3.1s)

&nbsp;&nbsp;scale 0.33  (3.4s)


WAVs dumped to /content/ab and /content/drive/MyDrive/aegis-urdu-loanword/ab

What to look for:
 - PLAIN rows: L1 small AND still sounds like Aegis at scale 0.33.
 - LOAN rows:  the loanword should be clearer at 0.33 than at 0.0.
 - scale 0.33 muffled -> lower ALPHA / drop enc_p.proj / fewer steps (cell 7).


## 9 · Merge the adapter into the weights → Piper ONNX

In [19]:
import os, glob, gc, torch, re, pathlib, shutil, json
SCALE    = 0.33   # ship candidate from the cell 8 sweep (1.0 was over-cooked)
OUT_NAME = "ur_PK-aegis_female_loan-medium"
#   distinct name while iterating; rename to "ur_PK-aegis_female-medium" for the
#   final drop-in ship (phoneme_id_map is byte-identical, checked below).
OUT = f"/content/{OUT_NAME}.onnx"

# Merge from the checkpoint cell 7 saved -- NOT the live `model`. So this cell
# survives a cell-8 OOM (just re-run 5->7, then here) and running it twice is
# harmless.
# prefer the best-val_mel checkpoint ModelCheckpoint saved (save_top_k=1);
# the final step often over-fits the ~18-clip val split. Set USE_LAST=True to
# merge the last step instead.
USE_LAST = False
_best = sorted(glob.glob("/content/train/ckpts/epoch=*-step=*.ckpt"))
_last = "/content/train/ckpts/last.ckpt"
if not USE_LAST and _best:
    _ck = _best[-1]
elif os.path.exists(_last):
    _ck = _last
else:
    _cands = sorted(glob.glob("/content/train/ckpts/*.ckpt"))
    assert _cands, "no checkpoint in /content/train/ckpts -- run the training cell first"
    _ck = _cands[-1]
print("merging from", _ck, "(best val_mel)" if _ck in _best else "(last step)")

# free the training / cell-8 residue before building another model
for _n in ("trainer", "dm", "model", "renders", "pv", "_mc", "_co", "_orig"):
    globals().pop(_n, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from piper.train.vits.lightning import VitsModel
mm = VitsModel(num_symbols=CFG["num_symbols"], num_speakers=1, sample_rate=22050,
               batch_size=8, mos_metric="none")
inject_lora(mm.model_g, TARGETS, RANK, ALPHA)
_sd = torch.load(_ck, map_location="cpu")["state_dict"]
_miss, _unexp = mm.load_state_dict(_sd, strict=False)
_lk = [k for k in _sd if ".lora_up." in k]
_ml = [k for k in _miss if ".lora_" in k]
assert _lk and not _ml, f"checkpoint LoRA weights unusable ({len(_lk)} found, missing {_ml[:2]})"
print(f"loaded {len(_lk)} adapter tensors from the checkpoint")

_n = set_lora_scale(mm.model_g, SCALE)
merge_lora(mm.model_g)                 # mm.model_g is a plain VITS again
print(f"merged {_n} adapters at scale {SCALE}")

_clean = {k: v for k, v in mm.hparams.items()}
torch.save({"state_dict": mm.state_dict(), "hyper_parameters": _clean,
            "pytorch-lightning_version": "2.0.0"}, "/content/merged.ckpt")

_p = pathlib.Path("/content/piper1-gpl/src/piper/train/export_onnx.py")
_s = _p.read_text()
_s = re.sub(r"(dynamo=False,\s*)+", "", _s)          # strip any prior insert(s)
_s = _s.replace("torch.onnx.export(", "torch.onnx.export(dynamo=False, ", 1)
_p.write_text(_s)
print("export_onnx.py: dynamo=False set (once)")
!cd /content/piper1-gpl && python -m piper.train.export_onnx --checkpoint /content/merged.ckpt --output-file {OUT}
assert os.path.exists(OUT), 'ONNX export failed — see error above'

shutil.copy("/content/aegis.onnx.json", OUT + ".json")
_a = json.load(open(OUT + ".json"))["phoneme_id_map"]
_b = json.load(open("/content/aegis.onnx.json"))["phoneme_id_map"]
print("phoneme_id_map:", "OK — drop-in" if _a == _b else "DIVERGED — do not ship")

# dump straight to Drive so the result survives a disconnect / a flaky download
if "DRIVE" in globals():
    _od = DRIVE / "loan-out"; _od.mkdir(exist_ok=True)
    shutil.copy(OUT, _od / f"{OUT_NAME}.onnx")
    shutil.copy(OUT + ".json", _od / f"{OUT_NAME}.onnx.json")
    print("exported to", _od)


merging from /content/train/ckpts/epoch=19-step=800.ckpt (best val_mel)
loaded 37 adapter tensors from the checkpoint
merged 37 adapters at scale 0.33
export_onnx.py: dynamo=False set (once)
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/content/piper1-gpl/src/piper/train/export_onnx.py:92: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(dynamo=False, model=model_g,
/content/piper1-gpl/src/piper/train/vits/attentions.py:235: TracerWarning: Converting a tensor 

## 10 · Final listen (the exported ONNX) + download

In [20]:
from piper import PiperVoice
import numpy as np, pathlib, wave
from IPython.display import Audio, display, Markdown

OUT_NAME = globals().get("OUT_NAME", "ur_PK-aegis_female_loan-medium")
v2 = PiperVoice.load(f"/content/{OUT_NAME}.onnx")
DUMP = pathlib.Path("/content/final"); DUMP.mkdir(exist_ok=True)

for i, t in enumerate([
        "براہِ کرم اپنے موبائل کے کی پیڈ سے پرانا پن مٹا دیں۔",
        "اس ٹرانزیکشن کی تصدیق کے لیے ایک دبائیں۔",
        "آپ کے کریڈٹ کارڈ کا بیلنس صفر ہو چکا ہے۔",
        "نئی اسٹیٹمنٹ آپ کی ای میل پر بھیج دی گئی ہے۔",
        "درج کردہ تاریخ اور کریڈٹ کارڈ نمبر دوبارہ چیک کریں۔",
        "آج میں آپ کی کیا مدد کر سکتی ہوں؟"]):
    pcm = np.concatenate([np.frombuffer(x.audio_int16_bytes, dtype=np.int16)
                          for x in v2.synthesize(t)])
    with wave.open(str(DUMP / f"{i}.wav"), "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(22050)
        w.writeframes(pcm.tobytes())
    bad = "  ⚠ silent" if np.abs(pcm).max() < 8 else ""
    display(Markdown(f"`{t}`  ({len(pcm)/22050:.1f}s){bad}"))
    display(Audio(pcm, rate=22050))

if "DRIVE" in globals():
    import shutil
    (DRIVE / "loan-out").mkdir(exist_ok=True)
    for f in DUMP.glob("*.wav"):
        shutil.copy(f, DRIVE / "loan-out" / f.name)
    print("final WAVs ->", DRIVE / "loan-out")


`براہِ کرم اپنے موبائل کے کی پیڈ سے پرانا پن مٹا دیں۔`  (4.1s)

`اس ٹرانزیکشن کی تصدیق کے لیے ایک دبائیں۔`  (3.0s)

`آپ کے کریڈٹ کارڈ کا بیلنس صفر ہو چکا ہے۔`  (2.9s)

`نئی اسٹیٹمنٹ آپ کی ای میل پر بھیج دی گئی ہے۔`  (3.4s)

`درج کردہ تاریخ اور کریڈٹ کارڈ نمبر دوبارہ چیک کریں۔`  (4.0s)

`آج میں آپ کی کیا مدد کر سکتی ہوں؟`  (2.4s)

final WAVs -> /content/drive/MyDrive/aegis-urdu-loanword/loan-out


In [21]:
from google.colab import files
OUT_NAME = globals().get("OUT_NAME", "ur_PK-aegis_female_loan-medium")
# already copied to Drive/loan-out by cell 9; this is just the browser download
for _ext in (".onnx", ".onnx.json"):
    try:
        files.download(f"/content/{OUT_NAME}{_ext}")
    except Exception as e:
        print("download failed (grab it from Drive/loan-out):", e)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>